# Pipeline de Clasificación — Test Visual Completo

| Sección | Descripción |
|---|---|
| **Config** | rutas, umbrales, colores de portero |
| **Carga** | modelo YOLO, prototipos PKL, homografías |
| **detect_frame** | detección + clasificación v3 + porteros |
| **Visor interactivo** | slider temporal, bboxes coloreados, mapa |
| **Score analysis** | distribución de scores y margins por clase |
| **Unknowns** | inspección de detecciones no clasificadas |
| **GK map** | posición de porteros en el campo |

## Requisitos externos
- **Vídeo panorámico VEO** (`data/videos/veo_panoramico_banyoles_1aParte.mp4`) — descargar con `yt-dlp`
- **Modelo YOLO jugadores** (`runs/detect/modelo_players_v24_panoramic2/weights/best.pt`) — entrenar con `train_yolo.py`
- **Homografías**: si no existen en `data/calib_alpha1/`, carga automáticamente `data/example_banyoles/H_camA.npy` y `H_camB.npy`
- **PKL prototipos**: si no existen `prototypes_v3_day.pkl`/`night.pkl`, carga automáticamente `data/example_banyoles/prototypes_day.pkl`/`night.pkl`

In [ ]:
# ════════════════════════════════════════════════
#  CONFIGURACIÓN — edita aquí
# ════════════════════════════════════════════════
VIDEO_PATH    = "data/videos/veo_panoramico_banyoles_1aParte.mp4"
PKL_DAY_PATH  = "prototypes_v3_day.pkl"
PKL_NIGHT_PATH= "prototypes_v3_night.pkl"
MODEL_PATH    = "runs/detect/modelo_players_v24_panoramic2/weights/best.pt"
H_A_PATH      = "data/calib_alpha1/calib_camA_alpha1_homography.npy"
H_B_PATH      = "data/calib_alpha1/calib_camB_alpha1_homography.npy"
SAM_PATH      = None   # ej. 'sam2_t.pt'  — None para desactivar

CONF              = 0.40
HOME_ATTACKS_LEFT = True
FIELD_L           = 98.0
FIELD_A           = 61.0

# Color de camiseta del portero en hex (None = usar posición en área)
GK_HOME_COLOR = None   # ej. '#ffff00'
GK_AWAY_COLOR = None   # ej. '#ff6600'

CLS_COLOR = {
    'player_home': '#3b82f6',
    'player_away': '#ef4444',
    'referee':     '#f59e0b',
    'gk_home':     '#60a5fa',
    'gk_away':     '#fca5a5',
    'unknown':     '#6b7280',
}
# -- Fallbacks a datos de ejemplo ----------------------------------
import os as _os
if not _os.path.exists(PKL_DAY_PATH):   PKL_DAY_PATH   = "data/example_banyoles/prototypes_day.pkl"
if not _os.path.exists(PKL_NIGHT_PATH): PKL_NIGHT_PATH = "data/example_banyoles/prototypes_night.pkl"
if not _os.path.exists(H_A_PATH):        H_A_PATH       = "data/example_banyoles/H_camA.npy"
if not _os.path.exists(H_B_PATH):        H_B_PATH       = "data/example_banyoles/H_camB.npy"


In [2]:
%matplotlib inline
import sys, cv2, pickle, warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import ipywidgets as widgets
from IPython.display import display, clear_output
from pathlib import Path
from collections import defaultdict

warnings.filterwarnings('ignore')
sys.path.insert(0, str(Path.cwd()))

from pipeline_core import (
    preprocess_half, init_undistort_maps,
    get_grass_hue, get_torso_crop, get_player_color,
    get_hue_signature, get_hue_signature_masked,
    extract_sam_masks_batch,
    classify_player_v3, classify_player,
    hue_sig_distance, _HUE_BINS,
    pixel_to_field, esquinas_metros,
    _unify_seam, _global_nms,
    apply_gk_by_position, hex_to_lab,
    select_protos, draw_pitch_mpl,
    L_M, A_M, COLOR_MAP_HEX,
    CLASSIFY_V3_MAX_SCORE, CLASSIFY_V3_MARGIN,
)
from ultralytics import YOLO

plt.rcParams.update({
    'figure.facecolor': '#0f0f1a', 'axes.facecolor': '#16213e',
    'text.color': 'white', 'axes.labelcolor': '#aab',
    'xtick.color': '#556', 'ytick.color': '#556', 'axes.edgecolor': '#334',
})
print('\u2705 Imports OK')

✅ Imports OK


In [ ]:
# ─────────────────────────────────────────────────────
#  Cargar modelo, prototipos (día/noche), homografías
# ─────────────────────────────────────────────────────
print(' Cargando YOLO...')
model = YOLO(MODEL_PATH)

sam_model = None
if SAM_PATH:
    try:
        from ultralytics import SAM
        sam_model = SAM(SAM_PATH)
        print(f' SAM2 cargado: {SAM_PATH}')
    except Exception as e:
        print(f' SAM2 no disponible: {e}')

with open(PKL_DAY_PATH,   'rb') as f: protos_day   = pickle.load(f)
with open(PKL_NIGHT_PATH, 'rb') as f: protos_night = pickle.load(f)
print(f' PKL día: {list(protos_day.keys())}')
print(f' PKL noche: {list(protos_night.keys())}')

H_a = np.load(H_A_PATH)
H_b = np.load(H_B_PATH)

cap_info = cv2.VideoCapture(VIDEO_PATH)
FPS     = cap_info.get(cv2.CAP_PROP_FPS) or 30.0
VID_W   = int(cap_info.get(cv2.CAP_PROP_FRAME_WIDTH))
VID_H   = int(cap_info.get(cv2.CAP_PROP_FRAME_HEIGHT))
DUR_S   = cap_info.get(cv2.CAP_PROP_FRAME_COUNT) / FPS
cap_info.release()

HALF_H = VID_H // 2
init_undistort_maps(half_h=HALF_H, vid_w=VID_W)

corners_a = esquinas_metros(H_a, VID_W, HALF_H)
corners_b = esquinas_metros(H_b, VID_W, HALF_H)
LIMITE_X  = float(np.mean([corners_a[:, 0].max(), corners_b[:, 0].min()]))

GK_HOME_LAB = hex_to_lab(GK_HOME_COLOR) if GK_HOME_COLOR else None
GK_AWAY_LAB = hex_to_lab(GK_AWAY_COLOR) if GK_AWAY_COLOR else None

print(f' {VID_W}×{VID_H} {FPS:.1f}fps {DUR_S/60:.1f}min')
print(f' Costura: {LIMITE_X:.1f}m | GK color: home={GK_HOME_COLOR} away={GK_AWAY_COLOR}')

In [ ]:
# ─────────────────────────────────────────────────────
#  detect_frame: detección + clasificación v3 + porteros
# ─────────────────────────────────────────────────────
def detect_frame(t_sec):
    """
    Devuelve (dets_a, dets_b, img_a_rgb, img_b_rgb, stats).
    stats: {n_a_raw, n_b_raw, n_seam_merged, n_nms_removed, n_final}
    """
    fidx = int(t_sec * FPS)
    cap  = cv2.VideoCapture(VIDEO_PATH)
    cap.set(cv2.CAP_PROP_POS_FRAMES, fidx)
    ret, frame = cap.read()
    cap.release()
    if not ret:
        return [], [], None, None, {}

    # Seleccionar PKL según iluminación del frame
    protos_cur = select_protos(frame, protos_day, protos_night)
    PROTOS_BASE = {k: v for k, v in protos_cur.items()
                   if k in ('player_home', 'player_away', 'referee')}

    raw_halves = []
    imgs = []

    for half_raw, cam_side, H in [
        (frame[:HALF_H, :], 'left',  H_a),
        (frame[HALF_H:, :], 'right', H_b),
    ]:
        cam_img = preprocess_half(half_raw, cam_side)
        gh      = get_grass_hue(cam_img) or 55.0
        results = model.predict(cam_img, conf=CONF, verbose=False)
        dets    = []

        if results and results[0].boxes is not None:
            boxes_np  = results[0].boxes.xyxy.cpu().numpy()
            confs_np  = results[0].boxes.conf.cpu().numpy()
            bboxes_l  = [(int(b[0]),int(b[1]),int(b[2]),int(b[3])) for b in boxes_np]
            sam_masks = extract_sam_masks_batch(cam_img, bboxes_l, sam_model)

            for idx_b, (box_t, conf_v) in enumerate(zip(boxes_np, confs_np)):
                x1, y1, x2, y2 = map(int, box_t)
                crop = cam_img[y1:y2, x1:x2]
                if crop.size == 0:
                    continue
                mask  = sam_masks[idx_b] if idx_b < len(sam_masks) else None
                torso = get_torso_crop(crop)

                cls, score, margin, _ = classify_player_v3(
                    crop, PROTOS_BASE, grass_hue=gh, mask=mask)

                _, color_mean, _ = get_player_color(torso)

                try:
                    pos_m = pixel_to_field((x1+x2)/2, float(y2), H)
                except Exception:
                    pos_m = None

                dets.append({
                    'cam':        cam_side,
                    'crop':       cv2.resize(crop, (52, 88)),
                    'crop_full':  crop,
                    'cls':        cls,
                    'score':      score,
                    'margin':     margin,
                    'conf_yolo':  float(conf_v),
                    'conf':       float(conf_v),
                    'pos_m':      pos_m,
                    'color_desc': color_mean.tolist(),
                    'bbox':       (x1, y1, x2, y2),
                    'gh':         gh,
                    'mask':       mask,
                    'img':        cam_img,
                })

        raw_halves.append(dets)
        imgs.append(cv2.cvtColor(cam_img, cv2.COLOR_BGR2RGB))

    n_a_raw = len(raw_halves[0])
    n_b_raw = len(raw_halves[1])

    after_seam = _unify_seam(
        [{**d} for d in raw_halves[0]],
        [{**d} for d in raw_halves[1]],
        cut_x=LIMITE_X)
    n_seam_merged = (n_a_raw + n_b_raw) - len(after_seam)

    unified = _global_nms(after_seam)
    n_nms_removed = len(after_seam) - len(unified)

    apply_gk_by_position(
        unified, HOME_ATTACKS_LEFT, GK_HOME_LAB, GK_AWAY_LAB,
        field_l=FIELD_L, field_a=FIELD_A)

    stats = {
        'n_a_raw':       n_a_raw,
        'n_b_raw':       n_b_raw,
        'n_seam_merged': n_seam_merged,
        'n_nms_removed': n_nms_removed,
        'n_final':       len(unified),
    }

    dets_a_out = [d for d in unified if d.get('cam') in ('left',  'AB')]
    dets_b_out = [d for d in unified if d.get('cam') in ('right', 'AB')]

    return dets_a_out, dets_b_out, imgs[0], imgs[1], stats


print(' detect_frame definido')

In [5]:
# ─────────────────────────────────────────────────────
#  Helpers de visualización
# ─────────────────────────────────────────────────────
def _hex_to_rgb01(h):
    h = h.lstrip('#')
    return tuple(int(h[i:i+2], 16)/255 for i in (0, 2, 4))

CLS_RGB01 = {k: _hex_to_rgb01(v) for k, v in CLS_COLOR.items()}


def draw_boxes_on_frame(img_rgb, dets):
    out = img_rgb.copy()
    for d in dets:
        x1, y1, x2, y2 = d['bbox']
        col = tuple(int(c*255) for c in CLS_RGB01.get(d['cls'], (0.5, 0.5, 0.5)))
        cv2.rectangle(out, (x1, y1), (x2, y2), col, 2)
        lbl = f"{d['cls'].split('_')[-1][0].upper()} {d['score']:.2f}/{d['margin']:.2f}"
        cv2.putText(out, lbl, (x1, max(y1-4, 0)),
                    cv2.FONT_HERSHEY_SIMPLEX, 0.42, col, 1, cv2.LINE_AA)
    return out


def show_frame_with_boxes(t_sec):
    clear_output(wait=True)
    dets_a, dets_b, img_a, img_b, stats = detect_frame(t_sec)
    if img_a is None:
        print('Sin frame'); return

    vis_a    = draw_boxes_on_frame(img_a, dets_a)
    vis_b    = draw_boxes_on_frame(img_b, dets_b)
    all_dets = dets_a + dets_b

    fig = plt.figure(figsize=(17, 9), facecolor='#0f0f1a')
    gs  = plt.GridSpec(2, 2, figure=fig,
                        width_ratios=[1.6, 1], height_ratios=[1, 1],
                        hspace=0.06, wspace=0.05)

    ax_a = fig.add_subplot(gs[0, 0])
    ax_a.imshow(vis_a); ax_a.axis('off')
    ax_a.set_title(f'Cámara A  ({stats.get("n_a_raw",0)} raw → {len(dets_a)} final)',
                   color='#aab', fontsize=9, pad=3)

    ax_b = fig.add_subplot(gs[1, 0])
    ax_b.imshow(vis_b); ax_b.axis('off')
    ax_b.set_title(f'Cámara B  ({stats.get("n_b_raw",0)} raw → {len(dets_b)} final)',
                   color='#aab', fontsize=9, pad=3)

    ax_m = fig.add_subplot(gs[:, 1])
    draw_pitch_mpl(ax_m, FIELD_L, FIELD_A)
    # línea de costura
    ax_m.axvline(LIMITE_X, color='#fde047', lw=1.2, ls='--', alpha=0.5, zorder=4)
    ax_m.axvspan(LIMITE_X - 2, LIMITE_X + 2, alpha=0.08, color='#fde047', zorder=3)

    counts = defaultdict(int)
    for d in all_dets:
        if d['pos_m'] is None: continue
        xm, ym = d['pos_m']
        if not (0 <= xm <= FIELD_L and 0 <= ym <= FIELD_A): continue
        col    = CLS_RGB01.get(d['cls'], (0.5, 0.5, 0.5))
        marker = 'D' if 'gk' in d['cls'] else 'o'
        ax_m.scatter(xm, ym, color=col, s=130, zorder=6,
                     marker=marker, edgecolors='white', linewidths=0.7)
        ax_m.text(xm, ym + 1.0, d['cls'].split('_')[-1][0].upper(),
                  color=col, fontsize=6.5, ha='center', zorder=7, fontweight='bold')
        counts[d['cls']] += 1

    patches = [mpatches.Patch(color=CLS_RGB01.get(cls, (0.5,0.5,0.5)),
                               label=f'{cls} ({counts[cls]})')
               for cls in CLS_COLOR if counts[cls] > 0]
    if patches:
        ax_m.legend(handles=patches, fontsize=7.5, loc='lower right',
                    facecolor='#0f0f1a', labelcolor='white',
                    framealpha=0.9, edgecolor='#334')

    seam_info = (f"costura −{stats.get('n_seam_merged',0)}  "
                 f"nms −{stats.get('n_nms_removed',0)}")
    ax_m.set_title(
        f"t={t_sec:.1f}s  |  {stats.get('n_a_raw',0)}+{stats.get('n_b_raw',0)} raw "
        f"→ {stats.get('n_final',0)} final  ({seam_info})",
        color='white', fontsize=8.5, fontweight='bold')

    try: plt.tight_layout()
    except Exception: pass
    plt.show()
    plt.close(fig)


print(' Helpers de visualización definidos')

✅ Helpers de visualización definidos


In [1]:
# ─────────────────────────────────────────────────────
#  Visor interactivo — slider temporal
# ─────────────────────────────────────────────────────
t_slider = widgets.FloatSlider(
    value=600.0, min=0.0, max=DUR_S, step=1.0,
    description='t (s):', continuous_update=False,
    style={'description_width': '55px'},
    layout=widgets.Layout(width='700px'))

out_widget = widgets.interactive_output(
    show_frame_with_boxes, {'t_sec': t_slider})

display(widgets.VBox([t_slider, out_widget]))

NameError: name 'widgets' is not defined

In [7]:
# ─────────────────────────────────────────────────────
#  Análisis de duplicados — costura + NMS
# ─────────────────────────────────────────────────────
DEDUP_START_S = 600
DEDUP_DUR_S   = 60
DEDUP_STEP_S  = 2.0

dedup_records = []
for t in np.arange(DEDUP_START_S, DEDUP_START_S + DEDUP_DUR_S, DEDUP_STEP_S):
    _, _, _, _, st = detect_frame(t)
    if st:
        dedup_records.append({**st, 't': t})

df_dedup = pd.DataFrame(dedup_records)
if len(df_dedup):
    total_raw  = (df_dedup['n_a_raw'] + df_dedup['n_b_raw']).sum()
    total_seam = df_dedup['n_seam_merged'].sum()
    total_nms  = df_dedup['n_nms_removed'].sum()
    total_fin  = df_dedup['n_final'].sum()
    print(f"{'Raw total':>18}: {total_raw}")
    print(f"{'−costura':>18}: {total_seam}  ({100*total_seam/max(total_raw,1):.1f}%)")
    print(f"{'−NMS':>18}: {total_nms}  ({100*total_nms/max(total_raw,1):.1f}%)")
    print(f"{'Final total':>18}: {total_fin}  ({100*total_fin/max(total_raw,1):.1f}%)")

    fig, axes = plt.subplots(1, 3, figsize=(15, 4), facecolor='#0f0f1a')

    # Evolución temporal
    ax = axes[0]
    ax.fill_between(df_dedup['t'], df_dedup['n_a_raw'] + df_dedup['n_b_raw'],
                    alpha=0.35, color='#94a3b8', label='raw A+B')
    ax.fill_between(df_dedup['t'], df_dedup['n_final'],
                    alpha=0.6, color='#22d3ee', label='final')
    ax.set_title('Detecciones por frame', color='white', fontsize=9)
    ax.set_xlabel('t (s)', color='#aab'); ax.set_facecolor('#0d0d1f')
    ax.legend(fontsize=8, facecolor='#0f0f1a', labelcolor='white')
    for sp in ax.spines.values(): sp.set_color('#334')

    # Eliminados por costura
    ax = axes[1]
    ax.bar(df_dedup['t'], df_dedup['n_seam_merged'],
           color='#fde047', alpha=0.8, width=DEDUP_STEP_S*0.85)
    ax.set_title('Eliminados por costura (unify_seam)', color='white', fontsize=9)
    ax.set_xlabel('t (s)', color='#aab'); ax.set_facecolor('#0d0d1f')
    ax.set_ylabel('nº duplicados', color='#aab')
    for sp in ax.spines.values(): sp.set_color('#334')

    # Eliminados por NMS
    ax = axes[2]
    ax.bar(df_dedup['t'], df_dedup['n_nms_removed'],
           color='#f87171', alpha=0.8, width=DEDUP_STEP_S*0.85)
    ax.set_title('Eliminados por NMS global', color='white', fontsize=9)
    ax.set_xlabel('t (s)', color='#aab'); ax.set_facecolor('#0d0d1f')
    ax.set_ylabel('nº duplicados', color='#aab')
    for sp in ax.spines.values(): sp.set_color('#334')

    fig.suptitle(
        f'Deduplicación — {DEDUP_DUR_S}s  |  '
        f'costura −{total_seam} ({100*total_seam/max(total_raw,1):.1f}%)  '
        f'NMS −{total_nms} ({100*total_nms/max(total_raw,1):.1f}%)',
        color='white', fontsize=10)
    try: plt.tight_layout()
    except Exception: pass
    plt.show(); plt.close(fig)
else:
    print('Sin datos')

         Raw total: 456
          −costura: 334  (73.2%)
              −NMS: 97  (21.3%)
       Final total: 25  (5.5%)


In [8]:
# ─────────────────────────────────────────────────────
#  Análisis de scores — procesa N segundos del vídeo
# ─────────────────────────────────────────────────────
SCAN_START_S = 600
SCAN_DUR_S   = 60
SCAN_STEP_S  = 2.0

records = []
for t in np.arange(SCAN_START_S, SCAN_START_S + SCAN_DUR_S, SCAN_STEP_S):
    dets_a, dets_b, _, _, _ = detect_frame(t)
    for d in dets_a + dets_b:
        records.append({'cls': d['cls'], 'score': d['score'], 'margin': d['margin']})

df_scores = pd.DataFrame(records)
print(f'Total detecciones: {len(df_scores)}')
if len(df_scores):
    print(df_scores['cls'].value_counts().to_string())

Total detecciones: 25
cls
unknown        15
referee         6
player_home     2
player_away     2


In [9]:
if len(df_scores) == 0:
    print('Sin datos')
else:
    classes = [c for c in CLS_COLOR if c in df_scores['cls'].values]
    fig, axes = plt.subplots(1, 2, figsize=(13, 4.5), facecolor='#0f0f1a')

    for cls in classes:
        sub = df_scores[df_scores['cls'] == cls]
        col = CLS_RGB01.get(cls, (0.5, 0.5, 0.5))
        axes[0].hist(sub['score'],  bins=30, alpha=0.65, color=col, label=cls, density=True)
        axes[1].hist(sub['margin'], bins=30, alpha=0.65, color=col, label=cls, density=True)

    axes[0].axvline(CLASSIFY_V3_MAX_SCORE, color='#fde047', ls='--', lw=1.5, label='umbral score')
    axes[1].axvline(CLASSIFY_V3_MARGIN,    color='#fde047', ls='--', lw=1.5, label='umbral margin')

    for ax, title in zip(axes, ['Score (Hellinger) — menor=mejor',
                                  'Margin (2\u00ba - 1\u00ba) — mayor=mejor']):
        ax.set_title(title, color='white', fontsize=9)
        ax.set_facecolor('#0d0d1f')
        ax.legend(fontsize=7.5, facecolor='#0f0f1a', labelcolor='white', framealpha=0.9)
        for sp in ax.spines.values(): sp.set_color('#334')

    fig.suptitle('Distribuci\u00f3n de scores de clasificaci\u00f3n', color='white', fontsize=11)
    try: plt.tight_layout()
    except Exception: pass
    plt.show(); plt.close(fig)

In [10]:
# ─────────────────────────────────────────────────────
#  Inspección de unknowns — galería de crops
# ─────────────────────────────────────────────────────
UNKNOWN_SCAN_START = 600
UNKNOWN_SCAN_DUR   = 30
UNKNOWN_STEP       = 1.5
MAX_SHOW           = 60

unknown_crops = []
for t in np.arange(UNKNOWN_SCAN_START, UNKNOWN_SCAN_START + UNKNOWN_SCAN_DUR, UNKNOWN_STEP):
    dets_a, dets_b, _, _, _ = detect_frame(t)
    for d in dets_a + dets_b:
        if d['cls'] == 'unknown':
            unknown_crops.append((d['crop'], d['score'], d['margin']))
    if len(unknown_crops) >= MAX_SHOW:
        break

print(f'Unknowns encontrados: {len(unknown_crops)}')

if unknown_crops:
    COLS   = 15
    rows_n = (len(unknown_crops) + COLS - 1) // COLS
    fig, axes = plt.subplots(rows_n, COLS,
                              figsize=(COLS * 0.7, rows_n * 1.4),
                              facecolor='#0f0f1a')
    if rows_n == 1: axes = axes[np.newaxis, :]
    for i, (crop, score, margin) in enumerate(unknown_crops):
        r, c = divmod(i, COLS)
        axes[r, c].imshow(cv2.cvtColor(crop, cv2.COLOR_BGR2RGB))
        axes[r, c].set_title(f'{score:.2f}\n{margin:.2f}',
                              fontsize=5.5, color='#fde047', pad=1)
        axes[r, c].axis('off')
    for i in range(len(unknown_crops), rows_n * COLS):
        r, c = divmod(i, COLS)
        axes[r, c].axis('off')
    fig.suptitle(f'Unknowns ({len(unknown_crops)})  score / margin',
                 color='white', fontsize=9)
    try: plt.tight_layout()
    except Exception: pass
    plt.show(); plt.close(fig)

Unknowns encontrados: 10


In [11]:
# ─────────────────────────────────────────────────────
#  Mapa de posición de porteros en el campo
# ─────────────────────────────────────────────────────
GK_SCAN_START = 600
GK_SCAN_DUR   = 120
GK_SCAN_STEP  = 3.0

gk_positions = defaultdict(list)
for t in np.arange(GK_SCAN_START, GK_SCAN_START + GK_SCAN_DUR, GK_SCAN_STEP):
    dets_a, dets_b, _, _, _ = detect_frame(t)
    for d in dets_a + dets_b:
        if 'gk' in d['cls'] and d['pos_m'] is not None:
            xm, ym = d['pos_m']
            if 0 <= xm <= FIELD_L and 0 <= ym <= FIELD_A:
                gk_positions[d['cls']].append((xm, ym))

fig, ax = plt.subplots(figsize=(11, 7), facecolor='#0f0f1a')
draw_pitch_mpl(ax, FIELD_L, FIELD_A)

for cls, pts in gk_positions.items():
    if not pts: continue
    xs, ys = zip(*pts)
    col = CLS_RGB01.get(cls, (0.5, 0.5, 0.5))
    ax.scatter(xs, ys, color=col, s=80, marker='D', zorder=6,
               edgecolors='white', linewidths=0.5,
               label=f'{cls} ({len(pts)} obs)', alpha=0.6)
    ax.scatter(np.mean(xs), np.mean(ys), color=col, s=320, marker='D',
               zorder=7, edgecolors='black', linewidths=1.5)

ax.legend(fontsize=9, facecolor='#0f0f1a', labelcolor='white',
          framealpha=0.9, edgecolor='#334')
ax.set_title(f'Posición de porteros — {GK_SCAN_DUR}s  (diamante grande = media)',
             color='white', fontsize=10, fontweight='bold')

if not any(gk_positions.values()):
    ax.text(FIELD_L/2, FIELD_A/2, 'No se detectaron porteros',
            color='#aab', ha='center', va='center', fontsize=12)

try: plt.tight_layout()
except Exception: pass
plt.show(); plt.close(fig)

for cls, pts in gk_positions.items():
    print(f'{cls}: {len(pts)} observaciones')